In [1]:
    !pip install -q timm albumentations
    
    import os
    import gc
    import random
    import numpy as np
    import pandas as pd
    
    from PIL import Image
    from tqdm import tqdm
    
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    
    from torch.utils.data import (
        Dataset,
        DataLoader,
        WeightedRandomSampler
    )
    
    import timm
    import albumentations as A
    from albumentations.pytorch import ToTensorV2
    
    from sklearn.metrics import f1_score
    
    ############################################################
    # SEED
    ############################################################
    
    SEED = 42
    
    random.seed(SEED)
    np.random.seed(SEED)
    
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    
    torch.backends.cudnn.benchmark = True
    
    DEVICE = (
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )
    
    print("DEVICE:", DEVICE)
    
    ############################################################
    # LOAD MANIFEST
    ############################################################
    
    MANIFEST_DIR = (
        "/kaggle/input/datasets/hritishachoudhury/bc-xai-quality-filtering"
    )
    
    df = pd.read_csv(
        f"{MANIFEST_DIR}/filtered_manifest.csv"
    )
    
    print("Manifest:", df.shape)
    
    ############################################################
    # LABELS
    ############################################################
    
    label_map = {
        "LumA":0,
        "LumB":1,
        "Her2":2,
        "Basal":3
    }
    
    df["label"] = (
        df["subtype_clean"]
        .map(label_map)
    )
    
    train_df = (
        df[df.split=="train"]
        .reset_index(drop=True)
    )
    
    val_df = (
        df[df.split=="val"]
        .reset_index(drop=True)
    )
    
    print("Train:", train_df.shape)
    print("Val:", val_df.shape)
    
    ############################################################
    # DATASET ROOTS
    ############################################################
    
    DATASET_ROOTS = {
    
        "bc-xai-a2-2000-partial":
        "/kaggle/input/datasets/anishapanja/bc-xai-a2-2000-partial",
    
        "bc-xai-e2-2000":
        "/kaggle/input/datasets/anishapanja/bc-xai-e2-2000",
    
        "bc-xai-a8-patches-batch1":
        "/kaggle/input/datasets/anishapanja/bc-xai-a8-patches-batch1",
    
        "bc-xai-a8-patches-batch2":
        "/kaggle/input/datasets/anishapanja/bc-xai-a8-patches-batch2",
    
        "bc-xai-a8-patches-batch3":
        "/kaggle/input/datasets/anishapanja/bc-xai-a8-patches-batch3",
    
        "bc-xai-a8-patches-batch4":
        "/kaggle/input/datasets/anishapanja/bc-xai-a8-patches-batch4",
    
        "bc-xai-a8-patches-batch5":
        "/kaggle/input/datasets/anishapanja/bc-xai-a8-patches-batch5",
    
        "bc-xai-c8-patches-batch1":
        "/kaggle/input/datasets/anishapanja/bc-xai-c8-patches-batch1",
    
        "bc-xai-c8-patches-batch2":
        "/kaggle/input/datasets/anishapanja/bc-xai-c8-patches-batch2",
    
        "bc-xai-c8-patches-batch3":
        "/kaggle/input/datasets/anishapanja/bc-xai-c8-patches-batch3",
    
        "bc-xai-c8-patches-batch4":
        "/kaggle/input/datasets/anishapanja/bc-xai-c8-patches-batch4",
    
        "bc-xai-bh-patches":
        "/kaggle/input/datasets/hritishachoudhury/bc-xai-bh-patches",
    
        "bc-xai-d8-patches":
        "/kaggle/input/datasets/hritishachoudhury/bc-xai-d8-patches"
    }
    
    ############################################################
    # PATH FUNCTION
    ############################################################
    
    def get_path(row):
    
        root = DATASET_ROOTS[
            row["dataset"]
        ]
    
        return os.path.join(
            root,
            row["relative_path"]
        )
    
    ############################################################
    # AUGMENTATIONS
    ############################################################
    
    train_tfms = A.Compose([
    
        A.RandomResizedCrop(
            size=(300,300),
            scale=(0.8,1.0)
        ),
    
        A.HorizontalFlip(p=0.5),
    
        A.VerticalFlip(p=0.5),
    
        A.Rotate(limit=90,p=0.5),
    
        A.ColorJitter(
            brightness=0.2,
            contrast=0.2,
            saturation=0.2,
            hue=0.1,
            p=0.7
        ),
    
        A.RandomGamma(
            gamma_limit=(80,120),
            p=0.4
        ),
    
        A.GaussNoise(p=0.3),
    
        A.GaussianBlur(p=0.2),
    
        A.Normalize(),
    
        ToTensorV2()
    
    ])
    
    val_tfms = A.Compose([
    
        A.Resize(300,300),
    
        A.Normalize(),
    
        ToTensorV2()
    
    ])
    
    ############################################################
    # DATASET
    ############################################################
    
    class BCDataset(Dataset):
    
        def __init__(
            self,
            df,
            transforms
        ):
    
            self.df = df
            self.transforms = transforms
    
        def __len__(self):
    
            return len(self.df)
    
        def __getitem__(self, idx):
    
            row = self.df.iloc[idx]
    
            try:
    
                img = Image.open(
                    get_path(row)
                ).convert("RGB")
    
            except:
    
                idx = np.random.randint(
                    len(self.df)
                )
    
                row = self.df.iloc[idx]
    
                img = Image.open(
                    get_path(row)
                ).convert("RGB")
    
            img = np.array(img)
    
            img = self.transforms(
                image=img
            )["image"]
    
            label = int(row.label)
    
            patient = row.patient_id
    
            return img,label,patient
    
    ############################################################
    # SAMPLER
    ############################################################
    
    counts = (
        train_df.label
        .value_counts()
    )
    
    weights = (
        len(train_df)
        /
        counts
    )
    
    sample_weights = (
        train_df.label
        .map(weights)
    )
    
    sampler = WeightedRandomSampler(
    
        sample_weights.values,
    
        num_samples=len(
            sample_weights
        ),
    
        replacement=True
    )
    
    ############################################################
    # DATALOADERS
    ############################################################
    
    train_ds = BCDataset(
        train_df,
        train_tfms
    )
    
    val_ds = BCDataset(
        val_df,
        val_tfms
    )
    
    train_loader = DataLoader(
    
        train_ds,
    
        batch_size=48,
    
        sampler=sampler,
    
        num_workers=2,
    
        pin_memory=True,
    
        persistent_workers=True
    )
    
    val_loader = DataLoader(
    
        val_ds,
    
        batch_size=48,
    
        shuffle=False,
    
        num_workers=2,
    
        pin_memory=True,
    
        persistent_workers=True
    )
    
    ############################################################
    # MODEL
    ############################################################
    
    model = timm.create_model(
    
        "efficientnet_b3",
    
        pretrained=True,
    
        num_classes=4,
    
        drop_rate=0.4
    )
    
    model = model.to(DEVICE)
    
    ############################################################
    # LOSS
    ############################################################
    
    class FocalLoss(nn.Module):
    
        def __init__(
            self,
            gamma=2
        ):
    
            super().__init__()
    
            self.gamma = gamma
    
        def forward(
            self,
            logits,
            targets
        ):
    
            ce = F.cross_entropy(
    
                logits,
    
                targets,
    
                reduction="none",
    
                label_smoothing=0.1
            )
    
            pt = torch.exp(-ce)
    
            loss = (
                (1-pt)**self.gamma
            ) * ce
    
            return loss.mean()
    
    
    criterion = FocalLoss()
    
    optimizer = torch.optim.AdamW(
    
        model.parameters(),
    
        lr=1e-4,
    
        weight_decay=5e-4
    )
    
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    
        optimizer,
    
        T_max=30
    )
    
    scaler = torch.cuda.amp.GradScaler()
    
    ############################################################
    # TRAINING
    ############################################################
    
    BEST = 0
    PATIENCE = 8
    counter = 0
    
    for epoch in range(30):
    
        print(f"\nEpoch {epoch+1}/30")
    
        #############################
        # TRAIN
        #############################
    
        model.train()
    
        losses = []
    
        for images,labels,_ in tqdm(
            train_loader
        ):
    
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)
    
            optimizer.zero_grad()
    
            with torch.cuda.amp.autocast():
    
                outputs = model(
                    images
                )
    
                loss = criterion(
                    outputs,
                    labels
                )
    
            scaler.scale(
                loss
            ).backward()
    
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                1.0
            )
    
            scaler.step(
                optimizer
            )
    
            scaler.update()
    
            losses.append(
                loss.item()
            )
    
        #############################
        # VALIDATION
        #############################
    
        model.eval()
    
        patient_probs = {}
    
        with torch.no_grad():
    
            for images,labels,patients in tqdm(
                val_loader
            ):
    
                images = images.to(
                    DEVICE
                )
    
                logits = model(
                    images
                )
    
                probs = F.softmax(
                    logits,
                    dim=1
                )
    
                probs = probs.cpu().numpy()
    
                for p,prob in zip(
                    patients,
                    probs
                ):
    
                    if p not in patient_probs:
    
                        patient_probs[p] = []
    
                    patient_probs[p].append(
                        prob
                    )
    
        gt = (
            val_df
            .groupby(
                "patient_id"
            )
            .first()
        )
    
        preds = []
        targets = []
    
        for patient in patient_probs:
    
            avg = np.mean(
                patient_probs[patient],
                axis=0
            )
    
            pred = np.argmax(avg)
    
            target = (
                gt.loc[
                    patient,
                    "label"
                ]
            )
    
            preds.append(pred)
            targets.append(target)
    
        f1 = f1_score(
    
            targets,
    
            preds,
    
            average="macro"
        )
    
        print(
            "Train Loss:",
            round(
                np.mean(losses),
                4
            )
        )
    
        print(
            "Patient F1:",
            round(
                f1,
                4
            )
        )
    
        scheduler.step()
    
        if f1 > BEST:
    
            BEST = f1
            counter = 0
    
            torch.save(
    
                model.state_dict(),
    
                "best_effnet_b3.pth"
            )
    
            print(
                "Saved Best Model"
            )
    
        else:
    
            counter += 1
    
        if counter >= PATIENCE:
    
            print(
                "Early Stopping"
            )
    
            break
    
        gc.collect()
        torch.cuda.empty_cache()
    
    print(
        "\nBEST F1:",
        BEST
    )

DEVICE: cuda
Manifest: (75000, 9)
Train: (51245, 10)
Val: (10911, 10)


model.safetensors:   0%|          | 0.00/49.3M [00:00<?, ?B/s]

/tmp/ipykernel_24/3698887477.py:408: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()



Epoch 1/30


  0%|          | 0/1068 [00:00<?, ?it/s]/tmp/ipykernel_24/3698887477.py:439: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
100%|██████████| 228/228 [01:33<00:00,  2.45it/s]


...

Train Loss: 0.3732
Patient F1: 0.5742

Epoch 9/30


  0%|          | 0/1068 [00:00<?, ?it/s]/tmp/ipykernel_24/3698887477.py:439: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
100%|██████████| 228/228 [00:50<00:00,  4.50it/s]


Train Loss: 0.3606
Patient F1: 0.5385

Epoch 10/30


  0%|          | 0/1068 [00:00<?, ?it/s]/tmp/ipykernel_24/3698887477.py:439: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
100%|██████████| 228/228 [00:50<00:00,  4.49it/s]

Train Loss: 0.3465
Patient F1: 0.5213
Early Stopping

BEST F1: 0.5855412083532601
